# 🎓 EduGuard AI — Stage 2: Preprocessing, Grouped Splitting & Feature Governance

**Course / Project:** EduGuard AI (Final-Year Project)  
**Module:** `notebooks/02_data_preprocessing.ipynb`  
**Author:** EduGuard Engineering Team (Mentorship Stage 2)  
**Stack:** Python 3.11+, pandas, scikit-learn, pathlib  

---

## 🛡️ Core Pedagogical & Engineering Objectives
In real-world educational data mining, naive machine learning workflows lead to catastrophic evaluation failures, primarily due to **data leakage** and **algorithmic bias**. In this notebook, we establish an end-to-end, publication-grade preprocessing pipeline that strictly enforces three rules:

1. **Zero Student Leakage (Grouped Splitting):** Approximately 382 students appear in both the Mathematics and Portuguese cohorts. A standard random train/test split would place records of the *same student* in both training and test sets, inflating model metrics deceptively. We enforce `GroupShuffleSplit` on `student_key`.
2. **Zero Target Leakage:** Final grade $G3$ and any derived proxies (`at_risk`, `passed`) are strictly isolated from predictor matrices.
3. **Algorithmic Fairness Sequestration:** Demographic and socioeconomic identifiers (`sex`, `age`, `address`, `famsize`, `Pstatus`, `Medu`, `Fedu`, `Mjob`, `Fjob`, `guardian`) are excluded from model inputs and held aside exclusively for post-hoc fairness audits.

### Step 1: Import Modules & Central Configuration

In [1]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
ROOT_DIR = Path.cwd().resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

from src.config import (
    RANDOM_STATE,
    RISK_PASS_MARK,
    SENSITIVE_FEATURES,
    LEAKAGE_COLS,
    TARGET_COL,
    PROCESSED_DATA_DIR
)
from src.data_loader import load_raw_dataset
from src.preprocessing import clean, make_student_key, grouped_train_test_split, build_preprocessor
from src.features import add_engineered_features, get_feature_sets, make_targets, prepare_scenario_data

print(f"Project root loaded: {ROOT_DIR}")
print(f"Pass mark threshold: {RISK_PASS_MARK}/20.0")
print(f"Protected sensitive attributes count: {len(SENSITIVE_FEATURES)}")

Project root loaded: D:\eduguard-ai
Pass mark threshold: 10.0/20.0
Protected sensitive attributes count: 11


### Step 2: Data Loading & Ingestion
We load both the Mathematics (`student-mat.csv`) and Portuguese (`student-por.csv`) raw datasets from `data/raw/` (semicolon-delimited), assign the course identifier `subject`, and concatenate them.

In [2]:
mat_df = load_raw_dataset('mat').copy()
por_df = load_raw_dataset('por').copy()

mat_df['subject'] = 'math'
por_df['subject'] = 'portuguese'

combined_raw = pd.concat([mat_df, por_df], ignore_index=True)
combined_raw['is_simulated'] = False

print(f"Mathematics cohort records: {len(mat_df)}")
print(f"Portuguese cohort records:  {len(por_df)}")
print(f"Total combined enrollments: {len(combined_raw)}")

Mathematics cohort records: 395
Portuguese cohort records:  649
Total combined enrollments: 1044


### Step 3: Range Validation & Data Sanitation (`clean()`)
Before any splitting, we run `clean(df)` to verify:
- No exact duplicates exist.
- Missing values (if present) are audited and handled.
- Numerical values (grades, absences, survey ratings) fall strictly within valid biological and pedagogical boundaries.

In [3]:
cleaned_df, report = clean(combined_raw)

print("=== CLEANING AUDIT REPORT ===")
for key, val in report.items():
    print(f"  {key}: {val}")

=== CLEANING AUDIT REPORT ===
  initial_rows: 1044
  final_rows: 1044
  exact_duplicates_removed: 0
  missing_values_handled: 0
  range_violations_repaired: 0
  actions_taken: []
  status: CLEAN


### Step 4: Generating Deterministic Student Keys (`make_student_key()`)
Cortez & Silva (2008) noted that students taking both courses can be uniquely identified by the 13 attributes:
`['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'nursery', 'internet']`.

We hash these components to generate a 12-character SHA-256 student identifier.

In [4]:
cleaned_df['student_key'] = make_student_key(cleaned_df)

mat_keys = set(cleaned_df[cleaned_df['subject'] == 'math']['student_key'])
por_keys = set(cleaned_df[cleaned_df['subject'] == 'portuguese']['student_key'])
shared_keys = mat_keys.intersection(por_keys)

print(f"Unique student identities in Math:       {len(mat_keys)}")
print(f"Unique student identities in Portuguese: {len(por_keys)}")
print(f"Students taking BOTH subjects:           {len(shared_keys)}")

Unique student identities in Math:       391
Unique student identities in Portuguese: 637
Students taking BOTH subjects:           366


### Step 5: Leak-Free Grouped Splitting (`grouped_train_test_split()`)
#### ⚠️ The Fatal Flaw of Standard `train_test_split`:
If we used `train_test_split(df, test_size=0.2)` randomly, roughly 20% of the 382 shared students would have their Mathematics record in `train` and their Portuguese record in `test`.
Because a student's personal study habits, parental background, and academic ability are strongly correlated across courses, the model would memorize student identities, artificially boosting test metrics while failing on genuinely new students.

#### The Solution:
`GroupShuffleSplit` groups all records sharing the same `student_key` together, ensuring that if a student is in `train`, ALL their course records remain in `train`.

In [5]:
train_df, test_df = grouped_train_test_split(
    cleaned_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    group_col="student_key"
)

train_keys = set(train_df['student_key'])
test_keys = set(test_df['student_key'])
overlap = train_keys.intersection(test_keys)

print(f"Training set records: {len(train_df)} ({len(train_keys)} unique students)")
print(f"Testing set records:  {len(test_df)} ({len(test_keys)} unique students)")
print(f"Overlapping students: {len(overlap)}")

# Non-negotiable assertion
assert len(overlap) == 0, "CRITICAL ERROR: Data leakage detected across splits!"
print("✅ VERIFIED: Zero student identity leakage between train and test splits.")

Training set records: 841 (529 unique students)
Testing set records:  203 (133 unique students)
Overlapping students: 0
✅ VERIFIED: Zero student identity leakage between train and test splits.


### Step 6: Leakage-Safe Feature Engineering (`add_engineered_features()`)
We create educational metrics without using future information:
1. `alcohol_total`: Combined weekly alcohol consumption (`Dalc + Walc`).
2. `support_total`: Sum of active interventions (`schoolsup + famsup + paid`).
3. `high_absence`: Chronic absenteeism indicator (`absences >= 10`).
4. `study_time_ratio`: Focus balance (`studytime / (freetime + 1.0)`).
5. `failures_flag`: Prior setback indicator (`failures > 0`).
6. `grade_trend` (Scenario B only): Performance momentum (`G2 - G1`).

In [6]:
# Augment both splits independently
train_aug = add_engineered_features(train_df, include_trend=True)
test_aug = add_engineered_features(test_df, include_trend=True)

# Create targets
train_aug['at_risk'] = (train_aug[TARGET_COL] < RISK_PASS_MARK).astype(int)
test_aug['at_risk'] = (test_aug[TARGET_COL] < RISK_PASS_MARK).astype(int)

engineered_cols = ['alcohol_total', 'support_total', 'high_absence', 'study_time_ratio', 'failures_flag', 'grade_trend']
train_aug[engineered_cols].describe().round(2)

,alcohol_total,support_total,high_absence,study_time_ratio,failures_flag,grade_trend
count,841.00,841.00,841.00,841.00,841.00,841.00
mean,3.76,0.94,0.15,0.51,0.17,0.01
std,1.96,0.78,0.35,0.28,0.38,1.73
min,2.00,0.00,0.00,0.17,0.00,-12.00
25%,2.00,0.00,0.00,0.33,0.00,-1.00
50%,3.00,1.00,0.00,0.50,0.00,0.00
75%,5.00,1.00,0.00,0.67,0.00,1.00
max,10.00,3.00,1.00,2.00,1.00,11.00


### Step 7: Dual-Scenario Feature Segregation & Fairness Audit
We extract features for **Scenario A** (Early Warning, strictly no $G1/G2$) and **Scenario B** (Mid-Semester, with $G1/G2$).

We verify that:
1. Sensitive demographic traits are **strictly absent** from both feature matrices.
2. Target $G3$ and target proxies are **strictly absent**.

In [7]:
feature_sets = get_feature_sets()

features_a = feature_sets['scenario_a']['all']
features_b = feature_sets['scenario_b']['all']

print(f"Scenario A feature count (Early Warning): {len(features_a)}")
print(f"Scenario B feature count (Mid-Semester):  {len(features_b)}")

# Verify sensitive attributes are excluded
for s in SENSITIVE_FEATURES:
    assert s not in features_a, f"Fairness violation: {s} in Scenario A"
    assert s not in features_b, f"Fairness violation: {s} in Scenario B"
print("✅ VERIFIED: All sensitive demographic traits sequestered from feature matrices.")

# Verify target is excluded
assert TARGET_COL not in features_a and TARGET_COL not in features_b
print("✅ VERIFIED: Target variable G3 strictly excluded from feature matrices.")

Scenario A feature count (Early Warning): 26
Scenario B feature count (Mid-Semester):  29
✅ VERIFIED: All sensitive demographic traits sequestered from feature matrices.
✅ VERIFIED: Target variable G3 strictly excluded from feature matrices.


### Step 8: Preprocessing Pipeline & Unseen Category Defense
We assemble our scikit-learn `ColumnTransformer` via `build_preprocessor()`, fit it **only on training data**, and transform test data.

We also simulate a student with an **unseen categorical level** (e.g. unknown school) to prove that `OneHotEncoder(handle_unknown='ignore')` safely maps unknown categories to zeros rather than crashing in production.

In [8]:
X_train_a, y_train_reg, y_train_clf, num_cols_a, cat_cols_a = prepare_scenario_data(train_df, scenario='A')
X_test_a, y_test_reg, y_test_clf, _, _ = prepare_scenario_data(test_df, scenario='A')

preprocessor = build_preprocessor(num_cols_a, cat_cols_a)

# Fit STRICTLY on training split
preprocessor.fit(X_train_a)

# Transform test split
X_test_trans = preprocessor.transform(X_test_a)
print(f"X_train shape: {X_train_a.shape} -> Transformed: {preprocessor.transform(X_train_a).shape}")
print(f"X_test shape:  {X_test_a.shape} -> Transformed: {X_test_trans.shape}")

# STRESS TEST: Unseen category
unseen_sample = X_test_a.iloc[[0]].copy()
unseen_sample.loc[unseen_sample.index[0], 'school'] = 'NEW_INTERNATIONAL_ACADEMY'
unseen_sample.loc[unseen_sample.index[0], 'reason'] = 'SCHOLARSHIP_PROGRAM'

unseen_trans = preprocessor.transform(unseen_sample)
assert unseen_trans.shape[1] == X_test_trans.shape[1]
print("✅ VERIFIED: Unseen categorical levels handled safely without runtime errors.")

X_train shape: (841, 26) -> Transformed: (841, 39)
X_test shape:  (203, 26) -> Transformed: (203, 39)
✅ VERIFIED: Unseen categorical levels handled safely without runtime errors.


### Step 9: Save Processed Partitions for Stage 3 Modeling
Finally, we persist `train.csv` and `test.csv` in `data/processed/`. These artifacts are fully prepared for Stage 3 regression, classification, SHAP analysis, and fairness audits.

In [9]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

train_path = PROCESSED_DATA_DIR / "train.csv"
test_path = PROCESSED_DATA_DIR / "test.csv"

train_aug.to_csv(train_path, index=False)
test_aug.to_csv(test_path, index=False)

print(f"Saved training partition: {train_path} ({len(train_aug)} rows)")
print(f"Saved testing partition:  {test_path} ({len(test_aug)} rows)")
print("🎉 Stage 2 preprocessing and splitting workflow completed successfully!")

Saved training partition: D:\eduguard-ai\data\processed\train.csv (841 rows)
Saved testing partition:  D:\eduguard-ai\data\processed\test.csv (203 rows)
🎉 Stage 2 preprocessing and splitting workflow completed successfully!
